In [1]:
import utils
import pandas as pd
from pathlib import Path
from collections import defaultdict 
import json
import matplotlib.pyplot as plt

In [2]:
# make a dictionary with key = drug, values = targets
drug_table = pd.read_excel(Path("../data/drugs_with_targets.xlsx"))

drug_t = (
    drug_table
        .dropna(subset=[drug_table.columns[2], drug_table.columns[3]])
        .assign(Genes=lambda df: df.iloc[:, 3].str.split(";"))
        .explode("Genes")
        .assign(Genes=lambda df: df["Genes"].str.strip())
        .groupby(drug_table.columns[2])["Genes"]
        .unique()
        .apply(list)
        .to_dict()
)

In [3]:
summary_df = pd.read_excel("../data/drugs_for_each_case_in_one_sheet.xlsx")

In [4]:
#side effects

def diff(row):
    val_dict = set( drug_t.get(row['Drug'],[]))

    val_tabel = set(
        x.strip() for x in row['Control nodes'].split(',') if x.strip()
    )

    return ",".join(sorted(val_dict - val_tabel))

summary_df['Side targets'] = summary_df.apply(diff, axis=1)


In [5]:
without_side_effects = (
    summary_df[summary_df["Side targets"].str.strip() == ""]
    .groupby("Patient")["Drug"]
    .nunique()
)

print(len(without_side_effects))
print(without_side_effects)
print("The maximum number",without_side_effects.max())
print("The minimum number",without_side_effects.min())

151
Patient
01521666-f595-4074-aea1-f7ab78db062b    33
020fcc42-97ff-4959-b52f-f7f119c3f643    34
02654be6-2049-4000-a0ac-c26f7ba6f0c9    31
02d43414-c8c9-42d3-8009-f9fb164dc8e0    22
0317a370-6a1e-44bd-bfd0-81fd06ec56fb    34
                                        ..
f8e7cbd2-b54a-4f30-8fc1-a5c80bfad8b6    36
fcc54ed3-5ba9-461f-a3a5-72b8eb89e4da    38
fe09042c-c233-4f10-b8ad-d5da80aaf60d    26
fe179a60-5e81-48c1-b437-7c38ed910ba8    28
fffc1088-c5a6-46a0-b050-860184f6ded2    25
Name: Drug, Length: 151, dtype: int64
The maximum number 46
The minimum number 17


In [6]:
# find the number of drugs that can influence more than 5 controlled proteins (primary tumors)
df_primary_5 = summary_df[
    (summary_df["Patient"].isin(utils.list_primary)) & (summary_df["Side targets"]=="") &
    (summary_df["Patient proteins length"] >= 5)
]["Patient"].value_counts()

print(len(df_primary_5))
print(df_primary_5)
print("The maximum number",df_primary_5.max())
print("The minimum number",df_primary_5.min())

134
Patient
6508275a-e712-424e-bdaf-b9e1b07b0f95    16
21800024-cf76-4185-b57b-526539ccdba2    15
2afdb646-75ca-4bc9-9c12-30a27f994ecd    15
8b4a22e2-e19a-4f8c-9f8f-a19f69fc8cbf    15
bd7b2c7c-3a38-4a2b-b6eb-5f67572db883    15
                                        ..
dc3875c2-1a88-491c-bf98-fa344dd3b07e     8
50a6757f-496c-4d2d-9463-f948c09c28f6     7
600932fb-d31e-4b69-afe7-e3803513d71f     7
6bdfd682-d5da-403d-b901-9d6fec360d3a     7
b0717300-0735-4451-bba1-7ce0fc0d4ac6     7
Name: count, Length: 134, dtype: int64
The maximum number 16
The minimum number 7


In [7]:
# find the number of drugs that can influence more than 10 controlled proteins (recurrent tumors)
df_recurrent_5 = summary_df[
    (summary_df["Patient"].isin(utils.list_recurrent)) &  (summary_df["Side targets"]=="") &
    (summary_df["Patient proteins length"] >= 5)
]["Patient"].value_counts()


print(len(df_recurrent_5))
print(df_recurrent_5)
print("The maximum number",df_recurrent_5.max())
print("The minimum number",df_recurrent_5.min())

13
Patient
57c3b272-7dcb-4937-83c1-02a3694e701f    14
c12052d7-f5ae-4fd5-a36b-ac0ce21179a6    13
6dfd7934-3161-4384-be4d-93b14080ece8    11
f8e7cbd2-b54a-4f30-8fc1-a5c80bfad8b6    11
8223f42b-434d-4966-a123-3b8a947884bc    10
1142d18f-9cd9-41b4-a9f2-975730cb713a     9
b4919075-90a9-4c8a-82a0-a0b5f9944ad3     9
d44d370d-d86e-4006-9530-ab3442cb2848     9
0317a370-6a1e-44bd-bfd0-81fd06ec56fb     8
928160c7-f711-4b23-832c-8c12aea9fe85     8
ddd4c4a7-6057-4244-aa4d-ccffe6e1c95a     8
f0f7d061-ace0-4dbd-90ec-df8f5985a7a2     8
fe179a60-5e81-48c1-b437-7c38ed910ba8     8
Name: count, dtype: int64
The maximum number 14
The minimum number 8
